# Laboratório Prático: Classificação de Tumores Mamários
## Implementação do KNN e Curva de Validação em Python

**Disciplina:** ADS308  
**Alunos:** Bernardo Motta e Diego Araújo  
**Prof.:** João Vítor Rodrigues de Vasconcelos  
**Curso:** Engenharia de Computação / Análise e Desenvolvimento de Sistemas  

---

### Objetivos
- Importar e manipular o dataset Breast Cancer Wisconsin usando pandas
- Aplicar o pré-processamento obrigatório (StandardScaler)
- Construir uma Curva de Validação para encontrar o K ótimo
- Treinar o modelo KNN final e medir sua acurácia

## O Cenário

Trabalharemos com o dataset **Breast Cancer Wisconsin**, um dos mais clássicos de Machine Learning.

**Objetivo do Negócio (Saúde):** Prever se um tumor mamário é **Maligno** ou **Benigno** com base em imagens digitalizadas de biópsias.

**Amostras:** 569 pacientes  
**Features:** 30 atributos numéricos (ex: raio médio, textura, perímetro, área, suavidade da célula)  
**Target:** 0 para Maligno, 1 para Benigno

## Passo 1: Importando as Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Bibliotecas do Scikit-Learn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## Passo 2: Carregando os Dados no Pandas

In [ ]:
# 1. Carregar o objeto bruto
dataset = load_breast_cancer()

# 2. Criar o DataFrame com as 30 features
df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)

# 3. Adicionar a coluna alvo (Target)
df["target"] = dataset.target

# Visualizar as 5 primeiras linhas
display(df.head())

In [ ]:
# Informações gerais do dataset
print(f"Formato do DataFrame: {df.shape}")
print(f"\nDistribuição do Target:")
print(df["target"].value_counts())
print(f"\n0 = Maligno | 1 = Benigno")

## Passo 3: Separando a Matriz X e o Vetor y

In [ ]:
# X recebe todas as 30 colunas de características
X = dataset.data

# y recebe apenas o diagnóstico (0 ou 1)
y = dataset.target

## Passo 4: Divisão de Treino e Teste

In [ ]:
# Reservando 30% dos dados para testar o modelo depois
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.3,
    random_state=70,
    stratify=y  # Mantém a proporção de tumores
)

print(f"Amostras de Treino: {X_treino.shape[0]}")
print(f"Amostras de Teste: {X_teste.shape[0]}")

## Passo 5: Escalonamento (Feature Scaling)

O KNN usa a **Distância Euclidiana** para encontrar os vizinhos. No dataset, a coluna "Área" pode ter valores na casa dos 1000, enquanto a "Suavidade" tem valores na casa de 0.01. Sem escalonamento, o algoritmo achará que a "Área" é a única variável que importa.

In [ ]:
# Instanciando o escalonador
scaler = StandardScaler()

# Ajustamos (fit) e transformamos apenas os dados de treino!
X_treino_escalonado = scaler.fit_transform(X_treino)

# Para o teste, usamos apenas transform (usando a média
# e o desvio padrão aprendidos no treino)
X_teste_escalonado = scaler.transform(X_teste)

## Passo 6: Construindo a Curva de Validação

Vamos criar um loop que treina um KNN diferente para cada valor ímpar de K (de 1 a 49), calcula o erro e guarda em uma lista.

In [ ]:
taxas_de_erro = []
valores_k = range(1, 50, 2)  # K ímpares (1, 3, 5... 49)

for k in valores_k:
    # 1. Cria o modelo com o K atual
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    # 2. Treina o modelo
    knn_temp.fit(X_treino_escalonado, y_treino)
    # 3. Faz a previsão no conjunto de teste
    previsoes = knn_temp.predict(X_teste_escalonado)
    # 4. Calcula o erro (média de acertos invertida)
    erro = np.mean(previsoes != y_teste)
    taxas_de_erro.append(erro)

## Passo 7: Encontrando o Mínimo Global

In [ ]:
# Pega o índice numérico onde ocorreu o menor erro
indice_melhor_k = np.argmin(taxas_de_erro)

# Busca qual era o K correspondente a esse índice
k_otimo = valores_k[indice_melhor_k]
menor_erro = taxas_de_erro[indice_melhor_k]

print(f"O K ótimo encontrado foi: {k_otimo}")
print(f"Com taxa de erro de: {menor_erro:.4f}")

## Passo 8: Plotando a Curva (Visualização)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(valores_k, taxas_de_erro,
         color="blue", linestyle="dashed",
         marker="o", markerfacecolor="red", markersize=8)

plt.axvline(x=k_otimo, color="green", linestyle="--",
            label=f"K ótimo = {k_otimo}")
plt.title("Curva de Validação do KNN")
plt.xlabel("Valor de K")
plt.ylabel("Taxa de Erro Média")
plt.legend()
plt.grid(True)
plt.show()

## Passo 9: Treinamento Definitivo

In [ ]:
# Agora que sabemos o melhor K, criamos o modelo oficial
modelo_final = KNeighborsClassifier(n_neighbors=k_otimo)

# Treinamos com a base de treino escalonada
modelo_final.fit(X_treino_escalonado, y_treino)

# Geramos as previsões oficiais para a base de teste
previsoes_finais = modelo_final.predict(X_teste_escalonado)

## Passo 10: Avaliação de Desempenho

In [ ]:
# Comparamos as previsões do modelo com o gabarito real
acuracia = accuracy_score(y_teste, previsoes_finais)

print(f"Acurácia Final do Modelo: {acuracia * 100:.2f}%")

---

## Relatório Final — Questões para Análise

As respostas abaixo compõem o relatório entregável da atividade.

### 1. Impacto do Escalonamento

**Pergunta:** Explique por que o uso do StandardScaler foi imprescindível neste dataset. O que aconteceria com o cálculo da Distância Euclidiana se ele não fosse aplicado?

---

O StandardScaler foi imprescindível porque o dataset Breast Cancer Wisconsin possui features com escalas extremamente diferentes entre si. A coluna "mean area", por exemplo, apresenta valores na casa das centenas a milhares (ex: 386 a 1326), enquanto a coluna "mean smoothness" opera na faixa de 0.05 a 0.16. Como o algoritmo KNN calcula a Distância Euclidiana entre os pontos para determinar os vizinhos mais próximos, essa diferença de magnitude faz com que as features de valores absolutos maiores dominem completamente o cálculo da distância.

A fórmula da Distância Euclidiana é dada por:

$$d(p, q) = \sqrt{\sum_{i=1}^{n}(p_i - q_i)^2}$$

Sem o escalonamento, a diferença $(p_i - q_i)^2$ para a feature "Área" (com diferenças na ordem de centenas) seria numericamente milhões de vezes maior do que a diferença para a feature "Suavidade" (com diferenças na ordem de centésimos). Na prática, o modelo ignoraria quase completamente as 29 features de menor escala e basearia sua decisão quase exclusivamente na "Área" — o que produziria um classificador enviesado e com desempenho inferior.

O StandardScaler resolve isso transformando cada feature para que tenha média igual a 0 e desvio padrão igual a 1 (padronização Z-score). Dessa forma, todas as 30 features contribuem de maneira equilibrada no cálculo da distância, e o modelo consegue capturar padrões reais presentes em todas as variáveis.

### 2. Interpretação da Curva

**Pergunta:** Analise a sua Curva de Validação. Descreva o comportamento do erro quando K = 1 e quando K ≥ 35. Relacione esses extremos aos conceitos de Overfitting e Underfitting.

---

A Curva de Validação obtida apresenta o formato característico esperado. O K ótimo encontrado foi **K = 3**, com a menor taxa de erro de **0.0526** (aproximadamente 5,26% de erro).

**Comportamento em K = 1 (extremo esquerdo — Overfitting):** Quando K = 1, o modelo classifica cada nova amostra com base em apenas um único vizinho mais próximo. Isso torna o classificador extremamente sensível a ruídos e outliers presentes nos dados de treino. Se houver um único ponto de treino ruidoso (por exemplo, um tumor benigno com características atípicas que o fazem parecer maligno), o modelo classificará erroneamente qualquer amostra de teste que esteja próxima desse ponto. O modelo, nesse caso, está essencialmente "memorizando" os dados de treino em vez de aprender padrões gerais — o que caracteriza o fenômeno de **Overfitting**. A taxa de erro em K = 1 é, portanto, mais alta do que no ponto ótimo.

**Comportamento em K ≥ 35 (extremo direito — Underfitting):** Conforme K cresce para valores como 35, 39, 43 ou 49, o modelo passa a considerar um número muito grande de vizinhos para tomar sua decisão. Ao fazer isso, a fronteira de decisão se torna excessivamente suavizada e o modelo perde a capacidade de distinguir padrões locais relevantes. Em um cenário extremo, se K fosse igual ao número total de amostras de treino, o modelo simplesmente classificaria toda nova amostra como a classe majoritária (Benigno), ignorando completamente as features. Esse comportamento excessivamente conservador é o **Underfitting** — o modelo é "brando demais" e não consegue capturar a complexidade real dos dados. Na curva, observa-se que a taxa de erro tende a aumentar novamente nessa região.

O K = 3 representa o ponto de equilíbrio (o fundo do "vale" na curva), onde o modelo generaliza bem sem memorizar ruído nem ignorar padrões importantes.

### 3. Estratificação dos Dados

**Pergunta:** Qual é a função exata do parâmetro `stratify=y` no momento da divisão dos dados? Por que ignorá-lo em um contexto médico pode invalidar o seu modelo?

---

O parâmetro `stratify=y` na função `train_test_split` garante que a divisão entre conjunto de treino e conjunto de teste preserve exatamente a mesma proporção de classes do dataset original. No nosso caso, o dataset possui 357 amostras benignas (classe 1) e 212 amostras malignas (classe 0), o que equivale a aproximadamente 62,7% benignas e 37,3% malignas. Com a estratificação, tanto o conjunto de treino (398 amostras) quanto o conjunto de teste (171 amostras) mantêm essa mesma proporção de aproximadamente 63/37.

Sem o `stratify=y`, a divisão é feita de forma puramente aleatória, e existe o risco real de que uma das classes fique sub-representada ou super-representada em um dos conjuntos. Por exemplo, poderia acontecer de o conjunto de teste conter apenas 10% de tumores malignos em vez dos 37% reais. Nesse cenário, o modelo seria treinado e avaliado em condições que não refletem a distribuição real dos pacientes.

Em um contexto médico, isso é particularmente grave por dois motivos. Primeiro, a classe minoritária (tumores malignos) é justamente a classe mais crítica — um erro de classificação de um tumor maligno como benigno (falso negativo) pode significar que o paciente não receberá o tratamento necessário, com consequências potencialmente fatais. Se o conjunto de treino tiver poucos exemplos de tumores malignos, o modelo simplesmente não aprenderá a reconhecê-los bem. Segundo, a métrica de acurácia reportada no conjunto de teste seria enganosa: um modelo que acertasse todos os benignos e errasse todos os malignos poderia ainda assim ter uma acurácia artificialmente alta se o teste tivesse poucos malignos, mascarando um desempenho clínico péssimo.

### 4. Viabilidade Clínica

**Pergunta:** Considerando a acurácia final obtida (e a simplicidade do método de distâncias), o algoritmo KNN seria confiável, por si só, para dar o diagnóstico final a um paciente real? Justifique apontando as limitações do modelo.

---

Apesar de o modelo ter alcançado uma acurácia de **94,74%**, o que à primeira vista parece um resultado expressivo, o KNN **não seria confiável como ferramenta única de diagnóstico final** para um paciente real. Essa conclusão se apoia em diversas limitações técnicas e clínicas.

Em primeiro lugar, uma acurácia de 94,74% implica que aproximadamente **5,26% dos pacientes seriam diagnosticados incorretamente**. Em um contexto oncológico, isso é inaceitável quando se trata de decisões isoladas. Um falso negativo (tumor maligno classificado como benigno) poderia levar à ausência de tratamento e à progressão do câncer, enquanto um falso positivo (tumor benigno classificado como maligno) submeteria o paciente a procedimentos invasivos desnecessários, como cirurgias e quimioterapia, além do impacto psicológico.

Em segundo lugar, a acurácia sozinha não revela como os erros estão distribuídos entre as classes. Para um cenário clínico, métricas como sensibilidade (recall para a classe maligna), especificidade, e a matriz de confusão completa seriam fundamentais para entender se o modelo está errando mais nos casos malignos ou benignos — e essa distinção tem implicações clínicas completamente diferentes.

Em terceiro lugar, o KNN é um modelo baseado puramente em distância geométrica e não oferece nenhuma explicabilidade clínica. Ele não indica quais features foram mais relevantes para a decisão, não fornece uma probabilidade calibrada de malignidade e não gera um raciocínio interpretável que o médico possa validar. Na medicina, a interpretabilidade do modelo é essencial para que o profissional de saúde possa confiar na recomendação e integrá-la ao seu julgamento clínico.

Por fim, o modelo foi treinado com apenas 569 amostras e 30 features extraídas de um único tipo de exame (biópsia digitalizada). Na prática clínica real, o diagnóstico de câncer de mama envolve múltiplas fontes de informação: histórico familiar, exames de imagem (mamografia, ultrassom, ressonância), marcadores biológicos e a análise anatomopatológica feita por um médico especialista.

Portanto, o KNN poderia ser utilizado como uma **ferramenta auxiliar de triagem** — sinalizando casos que merecem atenção prioritária — mas jamais como substituto do diagnóstico médico completo. A decisão final deve sempre ser do profissional de saúde, apoiada por múltiplos exames e, idealmente, por modelos mais robustos e interpretáveis.